In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password="ganesha@03",
    host="localhost",
    port="5432",
    database="finguard_db",
)
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM transactions", engine)
print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [2]:
df['Hour'] = (df['Time'] // 3600) % 24

In [3]:
# Flag transactions happening in the high-fraud-rate window (hours 1-5)
df['Is_High_Risk_Hour'] = df['Hour'].isin([1, 2, 3, 4, 5]).astype(int)

print(df['Is_High_Risk_Hour'].value_counts())
print(df.groupby('Is_High_Risk_Hour')['Class'].mean() * 100)

Is_High_Risk_Hour
0    268568
1     16239
Name: count, dtype: int64
Is_High_Risk_Hour
0    0.139257
1    0.726646
Name: Class, dtype: float64


In [4]:
# Log-transform Amount (raw amount is heavily right-skewed — log helps models learn better)
df['Amount_Log'] = np.log1p(df['Amount'])  # log1p handles Amount=0 safely

# Z-score of amount (how many std deviations from the mean)
df['Amount_Zscore'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()

# Is this a "round number" amount? (fraud sometimes uses suspiciously round test amounts)
df['Is_Round_Amount'] = (df['Amount'] % 1 == 0).astype(int)

print(df[['Amount', 'Amount_Log', 'Amount_Zscore', 'Is_Round_Amount']].head(10))

   Amount  Amount_Log  Amount_Zscore  Is_Round_Amount
0  149.62    5.014760       0.244964                0
1    2.69    1.305626      -0.342474                0
2  378.66    5.939276       1.160684                0
3  123.50    4.824306       0.140534                0
4   69.99    4.262539      -0.073403                0
5    3.67    1.541159      -0.338556                0
6    4.99    1.790091      -0.333278                0
7   40.80    3.732896      -0.190107                0
8   93.20    4.545420       0.019392                0
9    3.68    1.543298      -0.338516                0


In [5]:
# Faster approach using searchsorted (vectorized, no full-table apply)
time_vals = df['Time'].values
window_start = time_vals - 3600  # 1 hour = 3600 seconds

# For each transaction, count how many transactions fall in [time - 3600, time)
lower_idx = np.searchsorted(time_vals, window_start, side='left')
upper_idx = np.searchsorted(time_vals, time_vals, side='left')

df['Txn_Count_Last_Hour'] = upper_idx - lower_idx

print(df[['Time', 'Txn_Count_Last_Hour', 'Class']].head(10))
print("\nAvg txn count in last hour - Legit vs Fraud:")
print(df.groupby('Class')['Txn_Count_Last_Hour'].mean())

   Time  Txn_Count_Last_Hour  Class
0   0.0                    0      0
1   0.0                    0      0
2   1.0                    2      0
3   1.0                    2      0
4   2.0                    4      0
5   2.0                    4      0
6   4.0                    6      0
7   7.0                    7      0
8   7.0                    7      0
9   9.0                    9      0

Avg txn count in last hour - Legit vs Fraud:
Class
0    7246.302478
1    6082.847561
Name: Txn_Count_Last_Hour, dtype: float64


In [6]:
# Drop columns we no longer need in raw form
# (keep Amount for reference in EDA, but Amount_Log is what we'll feed models)
df_final = df.drop(columns=['Time'])  # raw Time no longer needed, we extracted Hour from it

print(df_final.columns.tolist())
print(df_final.shape)

['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class', 'Hour', 'Is_High_Risk_Hour', 'Amount_Log', 'Amount_Zscore', 'Is_Round_Amount', 'Txn_Count_Last_Hour']
(284807, 36)


In [7]:
# Save locally as CSV (for fast reloading in later notebooks)
df_final.to_csv("../data/processed/transactions_features.csv", index=False)
print("Saved to data/processed/transactions_features.csv")

# Also push to PostgreSQL as a new table
df_final.to_sql("transactions_features", engine, if_exists="replace", index=False, chunksize=10000)
print("Saved to PostgreSQL as 'transactions_features'")

Saved to data/processed/transactions_features.csv
Saved to PostgreSQL as 'transactions_features'
